In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframe
orders_df = pd.DataFrame(pd.DataFrame(datasets["olist_orders_dataset"]))
orders_df.shape

(99441, 8)

In [6]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [7]:
#Função para identificar valores nulos e percentuais de valores nulos em cada coluna do dataset
def quality_issues(df):
    nulls = df.isna().sum()
    percentual_nulls = (nulls / len(df) * 100).round(2)

    resum = pd.DataFrame({
        'Coluna': df.columns,
        'missing': nulls,
        'percentual_missing': percentual_nulls.values
    })
    return resum.sort_values('missing', ascending=False).reset_index(drop=True)

orders_quality_issues = quality_issues(orders_df)
orders_quality_issues

,Coluna,missing,percentual_missing
0,order_delivered_customer_date,2965,2.98
1,order_delivered_carrier_date,1783,1.79
2,order_approved_at,160,0.16
3,order_id,0,0.00
4,order_purchase_timestamp,0,0.00
5,order_status,0,0.00
6,customer_id,0,0.00
7,order_estimated_delivery_date,0,0.00


In [8]:
orders_df['order_id'].duplicated().sum()

np.int64(0)

In [9]:
orders_df['order_status'].nunique()
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [10]:
cols_data = orders_df[['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']]

for cols in cols_data:
    cols_dtype = orders_df[cols].dtype
    print(f"Coluna: {cols} | Tipo de dado: {cols_dtype}")
    na_count = orders_df[cols].isna().sum()
    print(f"Valores nulos: {na_count}")
    date_min = orders_df[cols].min()
    print(f"Data mínima: {date_min}")
    date_max = orders_df[cols].max()
    print(f"Data máxima: {date_max}")
    print("-" * 50)


Coluna: order_purchase_timestamp | Tipo de dado: str
Valores nulos: 0
Data mínima: 2016-09-04 21:15:19
Data máxima: 2018-10-17 17:30:18
--------------------------------------------------
Coluna: order_approved_at | Tipo de dado: str
Valores nulos: 160
Data mínima: 2016-09-15 12:16:38
Data máxima: 2018-09-03 17:40:06
--------------------------------------------------
Coluna: order_delivered_carrier_date | Tipo de dado: str
Valores nulos: 1783
Data mínima: 2016-10-08 10:34:01
Data máxima: 2018-09-11 19:48:28
--------------------------------------------------
Coluna: order_delivered_customer_date | Tipo de dado: str
Valores nulos: 2965
Data mínima: 2016-10-11 13:46:32
Data máxima: 2018-10-17 13:22:46
--------------------------------------------------
Coluna: order_estimated_delivery_date | Tipo de dado: str
Valores nulos: 0
Data mínima: 2016-09-30 00:00:00


Data máxima: 2018-11-12 00:00:00


--------------------------------------------------


In [11]:
for column in [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]:
    print(f"\nPedidos com {column} nulo:")
    print(
        orders_df.loc[
            orders_df[column].isna(),
            ["order_status", column]
        ]
        .groupby("order_status")
        .size()
        .sort_values(ascending=False)
    )


Pedidos com order_approved_at nulo:
order_status
canceled     141
delivered     14
created        5
dtype: int64

Pedidos com order_delivered_carrier_date nulo:
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
dtype: int64

Pedidos com order_delivered_customer_date nulo:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
dtype: int64


In [12]:
delivered = orders_df["order_status"] == "delivered"

# Caso 1: delivered sem data de entrega ao cliente
case1 = orders_df[delivered & orders_df["order_delivered_customer_date"].isna()]

# Caso 2: delivered sem data de aprovação
case2 = orders_df[delivered & orders_df["order_approved_at"].isna()]

print(f"Caso 1: {len(case1)} pedidos")
print(f"Caso 2: {len(case2)} pedidos")

print("\nExibindo os 5 primeiros pedidos do Caso 1:")
case1.head()

print("\nExibindo os 5 primeiros pedidos do Caso 2:")
case2.head()

Caso 1: 8 pedidos
Caso 2: 14 pedidos

Exibindo os 5 primeiros pedidos do Caso 1:

Exibindo os 5 primeiros pedidos do Caso 2:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00


In [13]:
p = orders_df["order_purchase_timestamp"]
a = orders_df["order_approved_at"]
c = orders_df["order_delivered_carrier_date"]
e = orders_df["order_delivered_customer_date"]

regras = {
    "aprovacao < compra": a < p,
    "envio_transportadora < compra": c < p,
    "entrega < envio_transportadora": e < c,
    "entrega < compra": e < p,
}

# Contagem por regra
resumo = pd.Series({nome: mask.sum() for nome, mask in regras.items()})
print(resumo)

# Pedidos que violam mais de uma regra
flags = pd.DataFrame(regras)
print("\nViolam 2+ regras:", (flags.sum(axis=1) >= 2).sum())

# Inspecionar cada violação
for nome, mask in regras.items():
    print(f"\n=== {nome}: {mask.sum()} ===")
    if mask.sum() > 0:
        display(orders_df.loc[mask, ["order_id", "order_status"] + list(cols_data)].head(10))

aprovacao < compra                  0
envio_transportadora < compra     166
entrega < envio_transportadora     23
entrega < compra                    0
dtype: int64

Violam 2+ regras: 0

=== aprovacao < compra: 0 ===

=== envio_transportadora < compra: 166 ===


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04 00:00:00
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18 00:00:00
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29 00:00:00
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08 00:00:00
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09 00:00:00
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36,2018-08-06 00:00:00
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43,2018-07-04 00:00:00
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27,2018-07-19 00:00:00
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53,2018-07-18 00:00:00
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaN,2018-05-28 00:00:00



=== entrega < envio_transportadora: 23 ===


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14 00:00:00
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21 00:00:00
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08 00:00:00
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11 00:00:00
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31 00:00:00
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30 00:00:00
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14 00:00:00
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25 00:00:00
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24 00:00:00
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18 00:00:00



=== entrega < compra: 0 ===


In [14]:
cols_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Cópia só para análise; orders_df continua intacto
datas = orders_df[cols_data].apply(pd.to_datetime, errors="coerce")
datas["order_id"] = orders_df["order_id"]
datas["order_status"] = orders_df["order_status"]

p = datas["order_purchase_timestamp"]
a = datas["order_approved_at"]
c = datas["order_delivered_carrier_date"]
e = datas["order_delivered_customer_date"]

In [15]:
g1 = datas[c < p].copy()
print("Total:", len(g1))

# Status
print(g1["order_status"].value_counts())
print("delivered:", (g1["order_status"] == "delivered").sum())

# Outras datas preenchidas
print(g1[cols_data].notna().sum())
print("Todas preenchidas:", g1[cols_data].notna().all(axis=1).sum())

Total: 166
order_status
delivered    165
shipped        1
Name: count, dtype: int64
delivered: 165
order_purchase_timestamp         166
order_approved_at                166
order_delivered_carrier_date     166
order_delivered_customer_date    165
order_estimated_delivery_date    166
dtype: int64
Todas preenchidas: 165


In [16]:
dif1 = (g1["order_delivered_carrier_date"] - g1["order_purchase_timestamp"])

print("Menor diferença:", dif1.min())
print("Maior diferença:", dif1.max())
print("Média:", dif1.mean())
print("Mediana:", dif1.median())

dif1_horas = dif1.dt.total_seconds() / 3600  # negativo = "voltou no tempo"
dif1_abs_horas = dif1_horas.abs()

ate_1h = (dif1_abs_horas <= 1).sum()
entre_1_24h = ((dif1_abs_horas > 1) & (dif1_abs_horas <= 24)).sum()
mais_24h = (dif1_abs_horas > 24).sum()

print(f"\nAté 1 hora: {ate_1h}")
print(f"Entre 1 e 24 horas: {entre_1_24h}")
print(f"Mais de 24 horas: {mais_24h}")
print(f"Total: {ate_1h + entre_1_24h + mais_24h} (deve bater com {len(g1)})")

Menor diferença: -172 days +18:54:07
Maior diferença: -1 days +23:59:36
Média: -2 days +21:57:37.879519
Mediana: -1 days +23:24:00.500000

Até 1 hora: 121
Entre 1 e 24 horas: 43
Mais de 24 horas: 2
Total: 166 (deve bater com 166)


In [17]:
g2 = datas[e < c].copy()
print("Total:", len(g2))
print(g2["order_status"].value_counts())

g2["dif_horas"] = (g2["order_delivered_customer_date"] - g2["order_delivered_carrier_date"]).dt.total_seconds() / 3600
print(g2["dif_horas"].describe())

print(g2[cols_data].notna().sum())
display(g2.sort_values("dif_horas").head(10))

# Overlap com os 166
print("Também nos 166:", g2["order_id"].isin(g1["order_id"]).sum())

Total: 23
order_status
delivered    23
Name: count, dtype: int64
count     23.000000
mean     -78.455133
std       89.308070
min     -386.308056
25%     -128.759028
50%      -39.864444
75%      -23.338333
max       -0.388333
Name: dif_horas, dtype: float64
order_purchase_timestamp         23
order_approved_at                23
order_delivered_carrier_date     23
order_delivered_customer_date    23
order_estimated_delivery_date    23
dtype: int64


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_id,order_status,dif_horas
34939,2017-02-10 10:19:10,2017-02-10 10:30:13,2017-03-02 17:34:26,2017-02-14 15:15:57,2017-03-15,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,-386.308056
27470,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,-189.095000
45302,2017-05-29 16:16:50,2017-05-29 16:25:16,2017-06-09 15:07:29,2017-06-02 11:09:16,2017-06-23,29941903985f944b0ffc49c479c1547d,delivered,-171.970278
14474,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11,dceb62e8fa94b46006c9554fed743df0,delivered,-144.238889
49933,2016-10-07 10:05:16,2016-10-07 11:24:43,2016-10-26 11:43:06,2016-10-20 18:03:17,2016-11-29,76458889992169d3135b264dc13aec67,delivered,-137.663611
71227,2016-10-07 17:09:56,2016-10-07 17:32:09,2016-10-26 11:42:05,2016-10-20 19:07:54,2016-12-01,19feb5627c41ea1b36a8e50a469b3644,delivered,-136.569722
74967,2017-02-04 19:01:33,2017-02-04 19:31:12,2017-02-15 08:55:26,2017-02-10 07:58:32,2017-03-01,d5558a097766363b8e76b38c43332e8a,delivered,-120.948333
41636,2017-01-27 14:59:17,2017-01-27 15:30:46,2017-02-20 02:32:08,2017-02-15 03:53:46,2017-04-17,b866af202be0692766081310cd4085e1,delivered,-118.639444
6437,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,-69.417222
78556,2017-07-18 13:38:29,2017-07-18 13:50:23,2017-07-27 19:21:31,2017-07-25 19:43:10,2017-08-15,ea1dcb4757a844d2642547797bd5feb0,delivered,-47.639167


Também nos 166: 0


In [18]:
dif2 = (g2["order_delivered_customer_date"] - g2["order_delivered_carrier_date"])

print("Menor diferença:", dif2.min())
print("Maior diferença:", dif2.max())
print("Média:", dif2.mean())
print("Mediana:", dif2.median())

dif2_horas = dif2.dt.total_seconds() / 3600
dif2_abs_horas = dif2_horas.abs()

ate_1h = (dif2_abs_horas <= 1).sum()
entre_1_24h = ((dif2_abs_horas > 1) & (dif2_abs_horas <= 24)).sum()
mais_24h = (dif2_abs_horas > 24).sum()

print(f"\nAté 1 hora: {ate_1h}")
print(f"Entre 1 e 24 horas: {entre_1_24h}")
print(f"Mais de 24 horas: {mais_24h}")
print(f"Total: {ate_1h + entre_1_24h + mais_24h} (deve bater com {len(g2)})")

Menor diferença: -17 days +21:41:31
Maior diferença: -1 days +23:36:42
Média: -4 days +17:32:41.521740
Mediana: -2 days +08:08:08

Até 1 hora: 2
Entre 1 e 24 horas: 5
Mais de 24 horas: 16
Total: 23 (deve bater com 23)


In [19]:
cols_ver = [
    "order_id", "order_status",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

pd.set_option("display.max_rows", 30)
display(g2[cols_ver].sort_values("order_delivered_customer_date"))

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-07 10:05:16,2016-10-07 11:24:43,2016-10-26 11:43:06,2016-10-20 18:03:17,2016-11-29
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-07 17:09:56,2016-10-07 17:32:09,2016-10-26 11:42:05,2016-10-20 19:07:54,2016-12-01
95634,f688669f48063536e082bb32d634cd46,delivered,2016-10-07 10:28:56,2016-10-11 04:56:13,2016-10-21 18:02:40,2016-10-20 20:33:27,2016-11-29
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-04 19:01:33,2017-02-04 19:31:12,2017-02-15 08:55:26,2017-02-10 07:58:32,2017-03-01
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-02-10 10:19:10,2017-02-10 10:30:13,2017-03-02 17:34:26,2017-02-14 15:15:57,2017-03-15
41636,b866af202be0692766081310cd4085e1,delivered,2017-01-27 14:59:17,2017-01-27 15:30:46,2017-02-20 02:32:08,2017-02-15 03:53:46,2017-04-17
76120,771c4f1f521f462e4b95619e648aaeab,delivered,2017-03-22 20:15:15,2017-03-22 20:15:15,2017-03-30 13:14:34,2017-03-28 17:28:32,2017-04-12
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-05-29 16:16:50,2017-05-29 16:25:16,2017-06-09 15:07:29,2017-06-02 11:09:16,2017-06-23
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14


In [20]:
total_pedidos = len(orders_df)
qtd_23 = len(g2)

print(f"Registros afetados: {qtd_23}")
print(f"Percentual do total: {qtd_23 / total_pedidos * 100:.4f}%")
print(f"\nDistribuição de status:\n{g2['order_status'].value_counts()}")
print(f"\ndelivered: {(g2['order_status'] == 'delivered').sum()}")
print(f"outros status: {(g2['order_status'] != 'delivered').sum()}")

Registros afetados: 23
Percentual do total: 0.0231%

Distribuição de status:
order_status
delivered    23
Name: count, dtype: int64

delivered: 23
outros status: 0


In [21]:
dif2 = g2["order_delivered_customer_date"] - g2["order_delivered_carrier_date"]
dif2_dias = dif2.dt.total_seconds() / 86400

print("Menor diferença (dias):", dif2_dias.min())
print("Maior diferença (dias):", dif2_dias.max())
print("Média (dias):", dif2_dias.mean())
print("Mediana (dias):", dif2_dias.median())

g2_ordenado = g2.assign(dif_dias=dif2_dias).sort_values("dif_dias")
display(g2_ordenado[cols_ver + ["dif_dias"]])

Menor diferença (dias): -16.096168981481483
Maior diferença (dias): -0.016180555555555556
Média (dias): -3.2689638687600646
Mediana (dias): -1.6610185185185184


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,dif_dias
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-02-10 10:19:10,2017-02-10 10:30:13,2017-03-02 17:34:26,2017-02-14 15:15:57,2017-03-15,-16.096169
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18,-7.878958
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-05-29 16:16:50,2017-05-29 16:25:16,2017-06-09 15:07:29,2017-06-02 11:09:16,2017-06-23,-7.165428
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11,-6.009954
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-07 10:05:16,2016-10-07 11:24:43,2016-10-26 11:43:06,2016-10-20 18:03:17,2016-11-29,-5.735984
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-07 17:09:56,2016-10-07 17:32:09,2016-10-26 11:42:05,2016-10-20 19:07:54,2016-12-01,-5.690405
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-04 19:01:33,2017-02-04 19:31:12,2017-02-15 08:55:26,2017-02-10 07:58:32,2017-03-01,-5.039514
41636,b866af202be0692766081310cd4085e1,delivered,2017-01-27 14:59:17,2017-01-27 15:30:46,2017-02-20 02:32:08,2017-02-15 03:53:46,2017-04-17,-4.943310
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14,-2.892384
78556,ea1dcb4757a844d2642547797bd5feb0,delivered,2017-07-18 13:38:29,2017-07-18 13:50:23,2017-07-27 19:21:31,2017-07-25 19:43:10,2017-08-15,-1.984965


In [22]:
qtd_166 = len(g1)
print(f"Registros afetados: {qtd_166}")
print(f"Percentual do total: {qtd_166 / total_pedidos * 100:.4f}%")
print(f"\nDistribuição de status:\n{g1['order_status'].value_counts()}")
print(f"\ndelivered: {(g1['order_status'] == 'delivered').sum()}")

dif1 = g1["order_delivered_carrier_date"] - g1["order_purchase_timestamp"]
dif1_horas = dif1.dt.total_seconds() / 3600

print("\nMenor diferença (horas):", dif1_horas.min())
print("Maior diferença (horas):", dif1_horas.max())

# Poucos minutos/horas vs vários dias
ate_1h = (dif1_horas.abs() <= 1).sum()
entre_1_24h = ((dif1_horas.abs() > 1) & (dif1_horas.abs() <= 24)).sum()
mais_24h = (dif1_horas.abs() > 24).sum()
print(f"\nAté 1h: {ate_1h} | 1-24h: {entre_1_24h} | >24h: {mais_24h}")

Registros afetados: 166
Percentual do total: 0.1669%

Distribuição de status:
order_status
delivered    165
shipped        1
Name: count, dtype: int64

delivered: 165

Menor diferença (horas): -4109.098055555555
Maior diferença (horas): -0.006666666666666667

Até 1h: 121 | 1-24h: 43 | >24h: 2


In [23]:
# Entre os inconsistentes dos 166 e dos 23, quantos entregaram depois do estimado?
atraso_166 = (g1["order_delivered_customer_date"] > g1["order_estimated_delivery_date"]).sum()
atraso_23 = (g2["order_delivered_customer_date"] > g2["order_estimated_delivery_date"]).sum()

print("Dos 166, entregues após o estimado:", atraso_166, f"de {g1['order_delivered_customer_date'].notna().sum()} com data de entrega preenchida")
print("Dos 23, entregues após o estimado:", atraso_23, f"de {qtd_23}")

Dos 166, entregues após o estimado: 3 de 165 com data de entrega preenchida
Dos 23, entregues após o estimado: 0 de 23


## 🧹 Tratamento — Orders

A partir daqui o notebook deixa de ser apenas profiling: aplica as decisões de tratamento sobre as inconsistências encontradas acima e salva o dataset limpo em `data/processed`.

In [24]:
#Fazendo copia do dataframe para não alterar o original
orders_clean = orders_df.copy()

In [25]:
# Convertendo as colunas de data de string para datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col])

orders_clean[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [26]:
# Regras de cronologia (antes de alterar qualquer valor)
purchase = orders_clean["order_purchase_timestamp"]
carrier = orders_clean["order_delivered_carrier_date"]
customer = orders_clean["order_delivered_customer_date"]

carrier_before_purchase = carrier < purchase
customer_before_carrier = customer < carrier
canceled_with_delivery = (orders_clean["order_status"] == "canceled") & customer.notna()

print(f"Envio à transportadora antes da compra: {carrier_before_purchase.sum()}")
print(f"Entrega ao cliente antes do envio à transportadora: {customer_before_carrier.sum()}")
print(f"Pedidos cancelados com data de entrega ao cliente: {canceled_with_delivery.sum()}")
print(f"Envio antes da compra E entrega antes do envio (sobreposição): {(carrier_before_purchase & customer_before_carrier).sum()}")

Envio à transportadora antes da compra: 166
Entrega ao cliente antes do envio à transportadora: 23
Pedidos cancelados com data de entrega ao cliente: 6
Envio antes da compra E entrega antes do envio (sobreposição): 0


In [27]:
# A data de envio à transportadora é a que contradiz a cronologia nos dois casos:
# a entrega ao cliente continua depois da compra, então o tempo de entrega (compra -> entrega) não é afetado.
# Por isso o valor inconsistente vira nulo (NaT), sem remover o pedido.
orders_clean.loc[carrier_before_purchase | customer_before_carrier, "order_delivered_carrier_date"] = pd.NaT

# Pedido cancelado não foi entregue: a data de entrega ao cliente vira nula
orders_clean.loc[canceled_with_delivery, "order_delivered_customer_date"] = pd.NaT

In [28]:
# Verificando quantos valores foram alterados em cada coluna
for col in orders_df.columns:
    before = pd.to_datetime(orders_df[col]) if col in date_cols else orders_df[col]
    after = orders_clean[col]
    changed = (before != after) & ~(before.isna() & after.isna())
    print(f"{col}: {changed.sum()} valores alterados")

order_id: 0 valores alterados


customer_id: 0 valores alterados
order_status: 0 valores alterados
order_purchase_timestamp: 0 valores alterados
order_approved_at: 0 valores alterados
order_delivered_carrier_date: 189 valores alterados
order_delivered_customer_date: 6 valores alterados
order_estimated_delivery_date: 0 valores alterados


In [29]:
# Casos mantidos sem alteração (documentados): delivered sem data de entrega / sem aprovação
delivered = orders_clean["order_status"] == "delivered"
print(f"delivered sem data de entrega ao cliente: {(delivered & orders_clean['order_delivered_customer_date'].isna()).sum()}")
print(f"delivered sem data de aprovação: {(delivered & orders_clean['order_approved_at'].isna()).sum()}")

delivered sem data de entrega ao cliente: 8
delivered sem data de aprovação: 14


In [30]:
# Validações pós-tratamento
p = orders_clean["order_purchase_timestamp"]
c = orders_clean["order_delivered_carrier_date"]
e = orders_clean["order_delivered_customer_date"]

print(f"Linhas: {orders_clean.shape[0]} (original: {orders_df.shape[0]})")
print(f"order_id duplicados: {orders_clean['order_id'].duplicated().sum()}")
print(f"Envio antes da compra: {(c < p).sum()}")
print(f"Entrega antes do envio: {(e < c).sum()}")
print(f"Entrega antes da compra: {(e < p).sum()}")
print(f"Pedidos cancelados com data de entrega: {((orders_clean['order_status'] == 'canceled') & e.notna()).sum()}")
print(f"customer_id sem correspondência em customers: {(~orders_clean['customer_id'].isin(datasets['olist_customers_dataset']['customer_id'])).sum()}")
print(f"Data mínima / máxima de compra: {p.min()} / {p.max()}")

assert orders_clean.shape == orders_df.shape
assert orders_clean['order_id'].is_unique
assert not (c < p).any() and not (e < c).any()
assert orders_clean['order_status'].equals(orders_df['order_status'])

Linhas: 99441 (original: 99441)
order_id duplicados: 0
Envio antes da compra: 0
Entrega antes do envio: 0
Entrega antes da compra: 0
Pedidos cancelados com data de entrega: 0


customer_id sem correspondência em customers: 0
Data mínima / máxima de compra: 2016-09-04 21:15:19 / 2018-10-17 17:30:18


In [31]:
# Salvando dataframe tratado em pasta processed
orders_clean.to_csv('../data/processed/olist_orders_dataset_clean.csv', index=False)

## 📌 Conclusão da etapa — Orders

### 🔎 O que foi encontrado?

A tabela `orders` possui **99.441 registros e 8 colunas**, sem duplicidade em `order_id`, com compras de **2016-09-04 a 2018-10-17**.

Durante o profiling, foram identificadas inconsistências nas datas:

- **166 pedidos** com envio à transportadora **antes** da compra;
- **23 pedidos** com entrega ao cliente **antes** do envio à transportadora (nenhum deles está entre os 166);
- **6 pedidos `canceled`** com data de entrega ao cliente;
- **8 pedidos `delivered`** sem data de entrega e **14** sem data de aprovação;
- Colunas de data armazenadas como `string`.

### 🛠️ O que foi feito?

- As 5 colunas de data foram convertidas para `datetime`;
- Nos 166 + 23 casos, a data de envio à transportadora (a que contradiz a cronologia) foi transformada em nulo, **189 valores alterados**. A entrega continua posterior à compra, então o tempo de entrega não é afetado;
- Nos 6 pedidos cancelados, a data de entrega ao cliente foi transformada em nulo;
- Os 8 `delivered` sem data de entrega e os 14 sem aprovação foram **mantidos**, sem inventar datas. Eles ficam fora dos cálculos de tempo de entrega por não terem data.

Nenhum registro foi removido e `order_status` não foi alterado.

### ✅ Como foi validado?

- A quantidade de registros permaneceu em **99.441**, sem `order_id` duplicado;
- Nenhum pedido com envio antes da compra, entrega antes do envio ou entrega antes da compra;
- Nenhum pedido cancelado com data de entrega;
- Todos os `customer_id` possuem correspondência em `customers`.

Nulos após o tratamento: `order_approved_at` 160, `order_delivered_carrier_date` 1.972 e `order_delivered_customer_date` 2.971.

### 📁 Resultado

O dataset tratado foi salvo em:

`data/processed/olist_orders_dataset_clean.csv`